# 6주차 강사용 Notebook — 공정별·설비별 데이터 비교

학생용 Notebook과 동일한 흐름이며, 예상 결과와 설명 포인트를 함께 담았습니다.

## 1단계. Pandas 불러오고 데이터 읽기

In [1]:
import pandas as pd

df = pd.read_csv("../../data/weekly/week06/week06_equipment_comparison.csv")
df.head()

,로트번호,설비번호,공정명,작업조,온도_섭씨,압력_Pa,진공도_mTorr,처리시간_sec,합격여부
0,LOT-0722,EQ-04,산화,C조,NaN,1003.7,5.63,117.2,1
1,LOT-0331,EQ-01,증착,B조,298.1,1010.0,5.28,116.5,1
2,LOT-0106,EQ-01,식각,C조,NaN,1003.7,4.69,123.8,1
3,LOT-0656,EQ-02,산화,A조,300.1,1013.7,5.05,117.5,1
4,LOT-0778,EQ-04,포토,B조,300.5,1007.6,5.05,128.9,-1


**예상 결과**: 9개 열(로트번호, 설비번호, 공정명, 작업조, 온도_섭씨, 압력_Pa, 진공도_mTorr, 처리시간_sec, 합격여부)이 보임.

## 2단계. 데이터 크기와 결측값 확인하기
**설명 포인트**: `온도_섭씨` 열에만 결측값이 있고, 전부 야간(C조) 기록임을 미리 언급한다.

In [2]:
print(df.shape)
print(df.isna().sum())

(240, 9)
로트번호          0
설비번호          0
공정명           0
작업조           0
온도_섭씨        10
압력_Pa         0
진공도_mTorr     0
처리시간_sec      0
합격여부          0
dtype: int64


**예상 결과**: (240, 9). `온도_섭씨` 열에 결측값 10건.

## 3단계. 설비별 평균 온도
**질문할 내용**: "이 결과만 보면 어떤 설비가 가장 정상적으로 보이나요?"

In [3]:
df.groupby("설비번호")["온도_섭씨"].mean().round(2)

설비번호
EQ-01    299.78
EQ-02    299.08
EQ-03    299.75
EQ-04    301.14
Name: 온도_섭씨, dtype: float64

**예상 결과**: EQ-02가 299.08℃로 가장 낮게 나와 얼핏 가장 정상적으로 보인다.

## 4단계. 설비별 온도 표준편차
**어려워할 부분**: 표준편차의 의미를 숫자로만 설명하면 어려워한다. 양궁 비유로 설명한다.

In [4]:
df.groupby("설비번호")["온도_섭씨"].std().round(2)

설비번호
EQ-01    2.39
EQ-02    7.16
EQ-03    2.67
EQ-04    4.12
Name: 온도_섭씨, dtype: float64

**예상 결과**: EQ-02의 표준편차가 7.16으로 다른 설비(2~4대)보다 훨씬 크다. 평균은 가장 낮았지만 실제로는 가장 불안정한 설비라는 점을 반드시 짚는다.

## 5단계. 설비별 평균 압력 & 공정별 평균 처리시간

In [5]:
df.groupby("설비번호")["압력_Pa"].mean().round(2)

설비번호
EQ-01    1009.18
EQ-02    1009.37
EQ-03    1027.39
EQ-04    1012.27
Name: 압력_Pa, dtype: float64

**예상 결과**: EQ-03이 1027.39 Pa로 다른 설비보다 약 18~20 Pa 높다(설계대로).

In [6]:
df.groupby("공정명")["처리시간_sec"].mean().round(2)

공정명
산화    118.80
세정    120.29
식각    119.11
증착    120.07
포토    119.97
Name: 처리시간_sec, dtype: float64

**예상 결과**: 공정별 평균 처리시간은 118.8~120.3초로 큰 차이가 없다.

## 6단계. 설비별·작업조별 불량률
**설명 포인트**: `(조건식).groupby(...).mean()`은 True/False를 1/0으로 취급하므로 그룹별 평균이 곧 비율이 된다는 점을 칠판에 적어가며 설명한다.

In [7]:
fail_rate_by_eq = (df["합격여부"] == -1).groupby(df["설비번호"]).mean() * 100
fail_rate_by_eq.round(1)

설비번호
EQ-01     6.5
EQ-02    10.5
EQ-03     7.0
EQ-04    21.9
Name: 합격여부, dtype: float64

**예상 결과**: EQ-04 21.9%로 가장 높음, EQ-01 6.5%로 가장 낮음.

In [8]:
fail_rate_by_shift = (df["합격여부"] == -1).groupby(df["작업조"]).mean() * 100
fail_rate_by_shift.round(1)

작업조
A조     8.1
B조    17.9
C조     9.2
Name: 합격여부, dtype: float64

**예상 결과**: B조가 17.9%로 가장 높다. A조 8.1%, C조 9.2%.

## 7단계. 가장 불안정한 설비 찾기
**설명 포인트**: `idxmax()`는 값이 가장 큰 항목의 '이름(인덱스)'을 돌려준다는 점을 `max()`와 비교해 설명한다.

In [9]:
most_unstable = df.groupby("설비번호")["온도_섭씨"].std().idxmax()
print("가장 불안정한 설비:", most_unstable)

가장 불안정한 설비: EQ-02


**예상 결과**: `EQ-02`. 6단계의 '불량률 1위(EQ-04)'와 다르다는 것이 오늘 수업의 핵심 결론이다.

## 8단계. 오류 대처 방법
- `TypeError: agg function failed` 발생 시: 문자열 열(로트번호 등)에 `.mean()`을 적용하려 한 경우다. 숫자 열만 집계해야 함을 안내.
- `KeyError` 발생 시: `df.columns`로 정확한 열 이름 확인.

## 확장 실습(빠른 학습자용)
설비 × 작업조 두 기준을 동시에 넣어 그룹화하면 어떤 결과가 나오는지 보여주면 7주차 예고가 된다.

In [10]:
df.groupby(["설비번호", "작업조"])["합격여부"].count()

설비번호   작업조
EQ-01  A조     16
       B조     22
       C조     24
EQ-02  A조     19
       B조     18
       C조     20
EQ-03  A조     24
       B조     17
       C조     16
EQ-04  A조     27
       B조     21
       C조     16
Name: 합격여부, dtype: int64